NB6 (fixed rerun): four-bucket evaluation with clean train/test separation.

What changed vs the original NB6:
   1. Buckets A and B are REMOVED from the training data (they were in it before).
   2. Bucket C is a RANDOM, deduplicated sample (before: first 500 rows in dataset order).
   3. Every rate is printed with a 95% confidence interval.
   4. Unused dataset loads (LMSYS, HackAPrompt) and the "train on the buckets" demo are gone.
   5. Predictions are saved (hashed IDs, no prompt text) for a future reproduction test.

In [5]:
import hashlib
import math
import os
import random
import re

import pandas as pd
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

SEED = 42
N_PER_BUCKET = 500
KEY_CHARS = 80  # prompts sharing their first 80 characters count as the same "template"


In [6]:
try:
    from google.colab import userdata
    hf_token = userdata.get("Token")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")




 Hugging Face login (WildChat is gated)

# Helpers

In [7]:
def norm(text):
    """Lowercase + collapse whitespace, so trivial differences don't hide duplicates."""
    return re.sub(r"\s+", " ", str(text).strip().lower())


def key(text):
    """Template key: the first KEY_CHARS characters of the normalized prompt."""
    return norm(text)[:KEY_CHARS]


def prompt_id(text):
    """Short hash so we can save predictions without saving prompt text."""
    return hashlib.sha1(norm(text).encode("utf-8")).hexdigest()[:12]


def wilson(k, n, z=1.96):
    """95% confidence interval for a rate k/n."""
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    denom = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return (centre - half, centre + half)



# load the training corpus

In [8]:
df = pd.read_csv("compressed_data.csv.gz")

# Bucket A / B: synthetic safe / unsafe (WildJailbreak)


In [9]:
wj = df[df["source"] == "wildjailbreak"]
bucket_a = wj[wj["label"] == "safe"].sample(N_PER_BUCKET, random_state=SEED)[["prompt"]].copy()
bucket_a["bucket"], bucket_a["true_label"] = "A_synthetic_safe", "safe"

bucket_b = wj[wj["label"] == "unsafe"].sample(N_PER_BUCKET, random_state=SEED)[["prompt"]].copy()
bucket_b["bucket"], bucket_b["true_label"] = "B_synthetic_unsafe", "unsafe"

# Bucket C: human safe (WildChat), RANDOM + deduplicated

In [10]:
wildchat = load_dataset("allenai/WildChat", split="train", token=hf_token)
candidates = wildchat.filter(
    lambda r: r["toxic"] == False
    and r["language"] == "English"
    and r["conversation"]
    and r["conversation"][0]["role"] == "user"
)
print("Qualifying WildChat conversations:", len(candidates))

rng = random.Random(SEED)
order = list(range(len(candidates)))
rng.shuffle(order)  # random order, fixed seed

seen_keys, c_prompts = set(), []
for i in order:
    p = candidates[i]["conversation"][0]["content"]
    k = key(p)
    if not p.strip() or k in seen_keys:
        continue  # skip empty prompts and repeats of the same template
    seen_keys.add(k)
    c_prompts.append(p)
    if len(c_prompts) == N_PER_BUCKET:
        break

bucket_c = pd.DataFrame({"prompt": c_prompts, "bucket": "C_human_safe", "true_label": "safe"})


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

data/train-00000-of-00006.parquet: reconstructing file:   0%|          |  0.00B /  291MB            

data/train-00000-of-00006.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00006.parquet: reconstructing file:   0%|          |  0.00B /  270MB            

data/train-00001-of-00006.parquet: downloading bytes:           |  0.00B            

...

data/train-00005-of-00006.parquet: reconstructing file:   0%|          |  0.00B /  231MB            

data/train-00005-of-00006.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/529428 [00:00<?, ? examples/s]

Filter:   0%|          | 0/529428 [00:00<?, ? examples/s]

Qualifying WildChat conversations: 284168


# Bucket D: human unsafe (TrustAIRLab jailbreaks)

In [11]:
bucket_d = df[(df["source"] == "trustairlab") & (df["label"] == "unsafe")][["prompt"]].copy()
bucket_d["bucket"], bucket_d["true_label"] = "D_human_unsafe", "unsafe"

nb6_df = pd.concat([bucket_a, bucket_b, bucket_c, bucket_d], ignore_index=True)
print("\nBucket sizes:\n", nb6_df["bucket"].value_counts())



Bucket sizes:
 bucket
D_human_unsafe        653
A_synthetic_safe      500
B_synthetic_unsafe    500
C_human_safe          500
Name: count, dtype: int64


# Training data: everything EXCEPT TrustAIRLab and the test buckets

In [12]:
held_out = set(nb6_df["prompt"].map(norm))
df_train = df[df["source"] != "trustairlab"]
df_train = df_train[~df_train["prompt"].map(norm).isin(held_out)].reset_index(drop=True)
print(f"\nTraining prompts after removing test buckets: {len(df_train):,}")


Training prompts after removing test buckets: 109,047


# Diagnostic: how many test prompts still share a template with a training prompt?

In [13]:
train_keys = set(df_train["prompt"].map(key))
print("\nShare of each bucket whose first-%d-chars match a training prompt:" % KEY_CHARS)
for name, g in nb6_df.groupby("bucket"):
    share = g["prompt"].map(key).isin(train_keys).mean()
    uniq = g["prompt"].map(key).nunique()
    print(f"  {name:20} {share:6.1%} overlap | {uniq} unique templates out of {len(g)}")


Share of each bucket whose first-80-chars match a training prompt:
  A_synthetic_safe       0.4% overlap | 500 unique templates out of 500
  B_synthetic_unsafe     0.2% overlap | 500 unique templates out of 500
  C_human_safe           0.2% overlap | 500 unique templates out of 500
  D_human_unsafe        13.5% overlap | 519 unique templates out of 653


# Train (same settings as the original NB6)

In [14]:
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train = vectorizer.fit_transform(df_train["prompt"])
clf = LogisticRegression(max_iter=1000, class_weight="balanced")
clf.fit(X_train, df_train["label"])


LogisticRegression(class_weight='balanced', max_iter=1000)

# Predict on the buckets

In [15]:
unsafe_col = list(clf.classes_).index("unsafe")
X_test = vectorizer.transform(nb6_df["prompt"])
nb6_df["predicted_label"] = clf.predict(X_test)
nb6_df["score"] = clf.predict_proba(X_test)[:, unsafe_col]

# Metrics with 95% confidence intervals

In [16]:
print("\n" + "=" * 72)
print("BUCKET RESULTS (clean rerun)")
print("=" * 72)
rates = {}
for name in ["A_synthetic_safe", "B_synthetic_unsafe", "C_human_safe", "D_human_unsafe"]:
    g = nb6_df[nb6_df["bucket"] == name]
    n = len(g)
    if name.endswith("_safe"):
        k = int((g["predicted_label"] == "unsafe").sum())
        label = "FP rate"
    else:
        k = int((g["predicted_label"] == "unsafe").sum())
        label = "Recall "
    lo, hi = wilson(k, n)
    rates[name] = k / n
    print(f"{name:20} | {label}: {k / n:6.1%}  (95% CI {lo:5.1%} to {hi:5.1%}) | n={n}")

gap = (rates["C_human_safe"] - rates["A_synthetic_safe"]) * 100
print(f"\nFP-rate gap, human safe minus synthetic safe: {gap:.1f} percentage points")



BUCKET RESULTS (clean rerun)
A_synthetic_safe     | FP rate:   1.8%  (95% CI  0.9% to  3.4%) | n=500
B_synthetic_unsafe   | Recall :  97.6%  (95% CI 95.9% to 98.6%) | n=500
C_human_safe         | FP rate:  34.0%  (95% CI 30.0% to 38.3%) | n=500
D_human_unsafe       | Recall :  63.6%  (95% CI 59.8% to 67.2%) | n=653

FP-rate gap, human safe minus synthetic safe: 32.2 percentage points


# Save predictions (hashed IDs, no prompt text)

In [18]:
import os
from google.colab import files

os.makedirs("results", exist_ok=True)
out = nb6_df.assign(prompt_id=nb6_df["prompt"].map(prompt_id))[
    ["prompt_id", "bucket", "true_label", "predicted_label", "score"]
]
out.to_csv("results/nb6_clean_predictions.csv", index=False)
print("\nSaved results/nb6_clean_predictions.csv")

# Download the file to your system
files.download("results/nb6_clean_predictions.csv")


Saved results/nb6_clean_predictions.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>